
# Phase 06 — Statistical Analysis & Experiment Design

## Project: Product Funnel and Conversion Drop-off Analysis

### 1. Objective

Evaluate statistical uncertainty, investigate associations between customer behavior and purchase participation, and develop a defensible A/B experiment proposal.

### 2. Statistical Research Questions

1. How precise are the observed customer purchase participation rates?
2. Is customer activity segment associated with purchase participation?
3. How large are the observed differences between customer segments?
4. Which business hypotheses could be evaluated through randomized experiments?
5. What sample size would a proposed A/B experiment require?

### 3. Methodological Principles

- Preserve the validated Phase 05 customer segmentation.
- Distinguish customer-level purchase participation from journey-level funnel conversion.
- Use appropriate statistical methods and verify assumptions.
- Account for repeated observations where necessary.
- Report confidence intervals and effect sizes alongside p-values.
- Do not interpret observational associations as causal effects.
- Clearly distinguish observed findings from experiment planning assumptions.

### 4. Analysis Roadmap

1. Statistical data readiness
2. Descriptive statistics
3. Confidence intervals
4. Hypothesis testing
5. Effect size analysis
6. Business hypothesis development
7. A/B experiment design
8. Statistical power and sample size
9. Final validation and documentation


In [2]:

# Step 06.5 — Statistical Data Readiness
# Inspect the existing cleaned dataset without loading all records.

from pathlib import Path
import pandas as pd

# Locate the cleaned dataset relative to the notebook.
project_root = Path.cwd().parent
clean_data_dir = project_root / "data" / "clean"

print("========== CLEANED DATASET INSPECTION ==========")

print(f"Project root: {project_root}")
print(f"Clean data directory exists: {clean_data_dir.exists()}")

clean_files = sorted(clean_data_dir.glob("*.csv"))

print(f"\nCSV files found: {len(clean_files)}")

for file in clean_files:
    print(f"\nFile: {file.name}")
    print(f"Size: {file.stat().st_size / (1024**2):.2f} MB")

    # Read only the header, not the full dataset.
    columns = pd.read_csv(file, nrows=0).columns.tolist()

    print(f"Columns: {columns}")

print("\n========== INSPECTION COMPLETE ==========")


========== CLEANED DATASET INSPECTION ==========
Project root: c:\Users\kunal\OneDrive\Desktop\product-funnel-analysis
Clean data directory exists: True

CSV files found: 1

File: october_customer_sample_clean.csv
Size: 112.37 MB
Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

========== INSPECTION COMPLETE ==========


In [3]:

# Step 06.6 — Inspect Cleaned Data Structure
# Read only five records to avoid unnecessary processing.

clean_file = clean_files[0]

sample_df = pd.read_csv(clean_file, nrows=5)

print("========== SAMPLE RECORDS ==========")
display(sample_df)

print("\n========== COLUMN DATA TYPES ==========")
print(sample_df.dtypes)

print("\n========== EVENT TYPES IN SAMPLE ==========")
print(sample_df["event_type"].unique())

print("\n========== INSPECTION COMPLETE ==========")


========== SAMPLE RECORDS ==========


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:56+0000,view,4804295,2053013554658804075,electronics.audio.headphone,xiaomi,23.13,541366014,6d8ce6fb-2953-4e95-96fe-b40f1ec50ba8
1,2019-10-01 00:03:35+0000,view,1005115,2053013555631882655,electronics.smartphone,apple,975.57,519829701,c3012f56-70f3-4198-960d-5f14467c0a53
2,2019-10-01 00:07:43+0000,view,15100370,2053013557024391671,Unknown,Unknown,257.15,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee
3,2019-10-01 00:08:33+0000,view,15100350,2053013557024391671,Unknown,Unknown,321.73,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee
4,2019-10-01 00:09:19+0000,view,15100100,2053013557024391671,Unknown,Unknown,437.33,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee



========== COLUMN DATA TYPES ==========
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object

========== EVENT TYPES IN SAMPLE ==========
<StringArray>
['view']
Length: 1, dtype: str

========== INSPECTION COMPLETE ==========


In [4]:

# Step 06.7 — Validate Event Distribution
# Read only the event_type column to minimize memory usage.

event_counts = pd.read_csv(
    clean_file,
    usecols=["event_type"]
)["event_type"].value_counts()

print("========== EVENT DISTRIBUTION ==========")

print(event_counts.to_string())

print("\n========== VALIDATION ==========")

total_events = int(event_counts.sum())

print(f"Total events: {total_events:,}")
print(f"Previously validated total: {849220:,}")

assert total_events == 849220, (
    "Event count does not match the Phase 05 validated total."
)

print("\nEVENT COUNT VALIDATION PASSED")


========== EVENT DISTRIBUTION ==========
event_type
view        814847
cart         18938
purchase     15435

========== VALIDATION ==========
Total events: 849,220
Previously validated total: 849,220

EVENT COUNT VALIDATION PASSED


In [5]:

# Step 06.8 — Validate Customer-Level Totals
# Read only customer IDs and event types.

customer_events = pd.read_csv(
    clean_file,
    usecols=["user_id", "event_type"]
)

total_customers = customer_events["user_id"].nunique()

purchasing_customers = customer_events.loc[
    customer_events["event_type"] == "purchase",
    "user_id"
].nunique()

print("========== CUSTOMER-LEVEL VALIDATION ==========")

print(f"Unique customers: {total_customers:,}")
print(f"Purchasing customers: {purchasing_customers:,}")

print("\n========== EXPECTED PHASE 05 TOTALS ==========")

print("Expected unique customers: 60,939")
print("Expected purchasing customers: 7,099")

assert total_customers == 60939, (
    "Unique customer count does not match Phase 05."
)

assert purchasing_customers == 7099, (
    "Purchasing customer count does not match Phase 05."
)

print("\nCUSTOMER-LEVEL VALIDATION PASSED")


========== CUSTOMER-LEVEL VALIDATION ==========
Unique customers: 60,939
Purchasing customers: 7,099

========== EXPECTED PHASE 05 TOTALS ==========
Expected unique customers: 60,939
Expected purchasing customers: 7,099

CUSTOMER-LEVEL VALIDATION PASSED


In [6]:

# Step 06.9 — Overall Customer Purchase Participation
# Unit of analysis: Unique customer

overall_purchase_participation = (
    purchasing_customers / total_customers
) * 100

print("========== OVERALL PURCHASE PARTICIPATION ==========")

print(f"Total customers: {total_customers:,}")
print(f"Purchasing customers: {purchasing_customers:,}")

print(
    f"Purchase participation: "
    f"{overall_purchase_participation:.2f}%"
)

print("\n========== METRIC DEFINITION ==========")

print("Unit of analysis: Unique customer")
print("Outcome: At least one purchase event")
print("This is NOT journey-level funnel conversion.")

print("\nOVERALL PURCHASE PARTICIPATION CALCULATED")


========== OVERALL PURCHASE PARTICIPATION ==========
Total customers: 60,939
Purchasing customers: 7,099
Purchase participation: 11.65%

========== METRIC DEFINITION ==========
Unit of analysis: Unique customer
Outcome: At least one purchase event
This is NOT journey-level funnel conversion.

OVERALL PURCHASE PARTICIPATION CALCULATED


In [7]:

# Step 06.10 — Wilson Confidence Interval
# Unit of analysis: Unique customer

from statsmodels.stats.proportion import proportion_confint

confidence_level = 0.95

ci_lower, ci_upper = proportion_confint(
    count=purchasing_customers,
    nobs=total_customers,
    alpha=1 - confidence_level,
    method="wilson"
)

print("========== PURCHASE PARTICIPATION ==========")

print(f"Total customers: {total_customers:,}")
print(f"Purchasing customers: {purchasing_customers:,}")

print(
    f"Observed participation: "
    f"{overall_purchase_participation:.2f}%"
)

print("\n========== 95% WILSON CONFIDENCE INTERVAL ==========")

print(f"Lower bound: {ci_lower * 100:.2f}%")
print(f"Upper bound: {ci_upper * 100:.2f}%")

print("\n========== METHODOLOGY ==========")
print("Unit: Unique customer")
print("Method: Wilson score interval")
print("Confidence level: 95%")
print("Assumption: Independent, representative customers")
print("Note: Sampling bias may limit population inference.")

print("\nCONFIDENCE INTERVAL CALCULATED")


========== PURCHASE PARTICIPATION ==========
Total customers: 60,939
Purchasing customers: 7,099
Observed participation: 11.65%

========== 95% WILSON CONFIDENCE INTERVAL ==========
Lower bound: 11.40%
Upper bound: 11.91%

========== METHODOLOGY ==========
Unit: Unique customer
Method: Wilson score interval
Confidence level: 95%
Assumption: Independent, representative customers
Note: Sampling bias may limit population inference.

CONFIDENCE INTERVAL CALCULATED


In [8]:

# Step 06.11 — Validate Customer Activity Segment Summary
# Reuse previously validated Phase 05 aggregate counts.
# Unit of analysis: Unique customer.

segment_stats = pd.DataFrame({
    "activity_segment": [
        "Low Activity",
        "Moderate Activity",
        "High Activity",
        "Very High Activity"
    ],
    "total_customers": [
        30626, 15371, 11902, 3040
    ],
    "purchasing_customers": [
        657, 1932, 3032, 1478
    ]
})

segment_stats["participation_rate"] = (
    segment_stats["purchasing_customers"]
    / segment_stats["total_customers"] * 100
)

print("========== CUSTOMER ACTIVITY SEGMENTS ==========")

display(segment_stats.round(2))

print("\n========== SEGMENT VALIDATION ==========")

assert segment_stats["total_customers"].sum() == total_customers

assert (
    segment_stats["purchasing_customers"].sum()
    == purchasing_customers
)

assert (
    segment_stats["purchasing_customers"]
    <= segment_stats["total_customers"]
).all()

print("Customer totals: PASSED")
print("Purchasing customer totals: PASSED")
print("Segment count consistency: PASSED")

print("\nSEGMENT VALIDATION COMPLETE")


========== CUSTOMER ACTIVITY SEGMENTS ==========


,activity_segment,total_customers,purchasing_customers,participation_rate
0,Low Activity,30626,657,2.15
1,Moderate Activity,15371,1932,12.57
2,High Activity,11902,3032,25.47
3,Very High Activity,3040,1478,48.62



========== SEGMENT VALIDATION ==========
Customer totals: PASSED
Purchasing customer totals: PASSED
Segment count consistency: PASSED

SEGMENT VALIDATION COMPLETE


In [9]:

# Step 06.12 — Segment-Level Wilson Confidence Intervals
# Unit of analysis: Unique customer
# Source: Validated Phase 05 segment aggregates

from statsmodels.stats.proportion import proportion_confint

segment_ci = segment_stats.copy()

# Calculate 95% Wilson intervals for each segment.
lower, upper = proportion_confint(
    count=segment_ci["purchasing_customers"],
    nobs=segment_ci["total_customers"],
    alpha=0.05,
    method="wilson"
)

segment_ci["ci_lower_pct"] = lower * 100
segment_ci["ci_upper_pct"] = upper * 100

print("========== SEGMENT CONFIDENCE INTERVALS ==========")

display(
    segment_ci[
        [
            "activity_segment",
            "total_customers",
            "purchasing_customers",
            "participation_rate",
            "ci_lower_pct",
            "ci_upper_pct"
        ]
    ].round(2)
)

print("\n========== VALIDATION ==========")

assert (
    segment_ci["ci_lower_pct"]
    <= segment_ci["participation_rate"]
).all()

assert (
    segment_ci["participation_rate"]
    <= segment_ci["ci_upper_pct"]
).all()

print("Confidence interval bounds: PASSED")
print("Method: Wilson score interval")
print("Confidence level: 95%")
print("Unit of analysis: Unique customer")
print("Population inference requires suitable sampling assumptions.")

print("\nSEGMENT CONFIDENCE INTERVALS CALCULATED")


========== SEGMENT CONFIDENCE INTERVALS ==========


,activity_segment,total_customers,purchasing_customers,participation_rate,ci_lower_pct,ci_upper_pct
0,Low Activity,30626,657,2.15,1.99,2.31
1,Moderate Activity,15371,1932,12.57,12.05,13.10
2,High Activity,11902,3032,25.47,24.70,26.27
3,Very High Activity,3040,1478,48.62,46.84,50.40



========== VALIDATION ==========
Confidence interval bounds: PASSED
Method: Wilson score interval
Confidence level: 95%
Unit of analysis: Unique customer
Population inference requires suitable sampling assumptions.

SEGMENT CONFIDENCE INTERVALS CALCULATED


In [10]:

# Step 06.13 — Chi-Square Test of Association
# Unit of analysis: Unique customer

from scipy.stats import chi2_contingency

# Build the customer-level contingency table.
contingency_table = segment_stats[
    ["purchasing_customers"]
].copy()

contingency_table["non_purchasing_customers"] = (
    segment_stats["total_customers"]
    - segment_stats["purchasing_customers"]
)

contingency_table.index = segment_stats["activity_segment"]

print("========== CONTINGENCY TABLE ==========")
display(contingency_table)

# Perform Pearson's chi-square test.
chi2_stat, p_value, degrees_freedom, expected = (
    chi2_contingency(
        contingency_table,
        correction=False
    )
)

print("\n========== CHI-SQUARE TEST ==========")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {degrees_freedom}")
print(f"P-value: {p_value:.6g}")

print("\n========== ASSUMPTION CHECK ==========")
print(f"Minimum expected frequency: {expected.min():.2f}")

assert (expected >= 5).all(), (
    "Expected frequency assumption requires review."
)

assert contingency_table.to_numpy().sum() == total_customers

print("Expected frequencies: PASSED")
print("Customer totals: PASSED")
print("Independence and sampling assumptions require review.")

print("\nCHI-SQUARE TEST CALCULATED")


========== CONTINGENCY TABLE ==========


,purchasing_customers,non_purchasing_customers
activity_segment,,
Low Activity,657,29969
Moderate Activity,1932,13439
High Activity,3032,8870
Very High Activity,1478,1562



========== CHI-SQUARE TEST ==========
Chi-square statistic: 8947.6337
Degrees of freedom: 3
P-value: 0

========== ASSUMPTION CHECK ==========
Minimum expected frequency: 354.14
Expected frequencies: PASSED
Customer totals: PASSED
Independence and sampling assumptions require review.

CHI-SQUARE TEST CALCULATED


In [11]:

# Step 06.14 — Cramér's V Effect Size
# Measure association between activity segment and purchase participation.

import numpy as np

n = contingency_table.to_numpy().sum()
rows, columns = contingency_table.shape

# Cramér's V formula
cramers_v = np.sqrt(
    chi2_stat / (n * min(rows - 1, columns - 1))
)

print("========== EFFECT SIZE ANALYSIS ==========")

print(f"Total customers: {n:,}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Cramér's V: {cramers_v:.4f}")

print("\n========== VALIDATION ==========")

assert 0 <= cramers_v <= 1, (
    "Cramér's V must be between 0 and 1."
)

assert n == total_customers

print("Effect size range: PASSED")
print("Customer denominator: PASSED")

print("\n========== INTERPRETATION ==========")

print("Cramér's V measures strength of association.")
print("It does not establish a causal relationship.")
print("Business importance requires separate evaluation.")

print("\nEFFECT SIZE CALCULATION COMPLETE")


========== EFFECT SIZE ANALYSIS ==========
Total customers: 60,939
Chi-square statistic: 8947.6337
Cramér's V: 0.3832

========== VALIDATION ==========
Effect size range: PASSED
Customer denominator: PASSED

========== INTERPRETATION ==========
Cramér's V measures strength of association.
It does not establish a causal relationship.
Business importance requires separate evaluation.

EFFECT SIZE CALCULATION COMPLETE


In [12]:

# Step 06.15 — Segment Comparison and Practical Effect Sizes
# Unit of analysis: Unique customer

low = segment_stats.loc[
    segment_stats["activity_segment"] == "Low Activity"
].iloc[0]

very_high = segment_stats.loc[
    segment_stats["activity_segment"] == "Very High Activity"
].iloc[0]

low_rate = (
    low["purchasing_customers"] / low["total_customers"]
)

very_high_rate = (
    very_high["purchasing_customers"]
    / very_high["total_customers"]
)

# Absolute difference in percentage points
absolute_difference_pp = (
    very_high_rate - low_rate
) * 100

# Relative participation ratio
participation_ratio = very_high_rate / low_rate

print("========== SEGMENT COMPARISON ==========")

print(f"Low Activity: {low_rate * 100:.2f}%")
print(f"Very High Activity: {very_high_rate * 100:.2f}%")

print("\n========== EFFECT SIZES ==========")

print(
    f"Absolute difference: "
    f"{absolute_difference_pp:.2f} percentage points"
)

print(
    f"Participation ratio: "
    f"{participation_ratio:.2f}x"
)

print("\n========== VALIDATION ==========")

assert very_high_rate > low_rate
assert participation_ratio > 1

print("Observed difference: PASSED")
print("Participation ratio: PASSED")

print("\n========== INTERPRETATION ==========")

print("These are observational group differences.")
print("They do not demonstrate causation.")
print("Activity may be associated with other customer characteristics.")

print("\nSEGMENT EFFECT SIZE COMPARISON COMPLETE")


========== SEGMENT COMPARISON ==========
Low Activity: 2.15%
Very High Activity: 48.62%

========== EFFECT SIZES ==========
Absolute difference: 46.47 percentage points
Participation ratio: 22.66x

========== VALIDATION ==========
Observed difference: PASSED
Participation ratio: PASSED

========== INTERPRETATION ==========
These are observational group differences.
They do not demonstrate causation.
Activity may be associated with other customer characteristics.

SEGMENT EFFECT SIZE COMPARISON COMPLETE


In [13]:

# Step 06.16 — Confidence Interval for Segment Difference
# Method: Newcombe confidence interval
# Unit of analysis: Unique customer

from statsmodels.stats.proportion import confint_proportions_2indep

difference_lower, difference_upper = (
    confint_proportions_2indep(
        count1=int(very_high["purchasing_customers"]),
        nobs1=int(very_high["total_customers"]),
        count2=int(low["purchasing_customers"]),
        nobs2=int(low["total_customers"]),
        compare="diff",
        method="newcomb",
        alpha=0.05
    )
)

print("========== SEGMENT DIFFERENCE ==========")

print(
    f"Observed difference: "
    f"{absolute_difference_pp:.2f} percentage points"
)

print("\n========== 95% CONFIDENCE INTERVAL ==========")

print(
    f"Lower bound: {difference_lower * 100:.2f} "
    "percentage points"
)

print(
    f"Upper bound: {difference_upper * 100:.2f} "
    "percentage points"
)

print("\n========== VALIDATION ==========")

assert (
    difference_lower * 100
    <= absolute_difference_pp
    <= difference_upper * 100
)

print("Confidence interval bounds: PASSED")
print("Method: Newcombe")
print("Confidence level: 95%")
print("Unit of analysis: Unique customer")
print("Assumes independent observations and suitable sampling.")

print("\nSEGMENT DIFFERENCE CONFIDENCE INTERVAL COMPLETE")


========== SEGMENT DIFFERENCE ==========
Observed difference: 46.47 percentage points

========== 95% CONFIDENCE INTERVAL ==========
Lower bound: 44.69 percentage points
Upper bound: 48.26 percentage points

========== VALIDATION ==========
Confidence interval bounds: PASSED
Method: Newcombe
Confidence level: 95%
Unit of analysis: Unique customer
Assumes independent observations and suitable sampling.

SEGMENT DIFFERENCE CONFIDENCE INTERVAL COMPLETE


In [14]:

# Step 06.17 — Business Hypothesis Development
# These are proposed experiments, not completed tests.

business_hypotheses = pd.DataFrame([
    {
        "hypothesis_id": "H1",
        "opportunity": "Low purchase participation",
        "proposed_change": "Simplify add-to-cart experience",
        "primary_metric": "Eligible user purchase conversion",
        "expected_direction": "Increase"
    },
    {
        "hypothesis_id": "H2",
        "opportunity": "Cart-to-purchase friction",
        "proposed_change": "Simplify checkout flow",
        "primary_metric": "Eligible cart-user purchase conversion",
        "expected_direction": "Increase"
    },
    {
        "hypothesis_id": "H3",
        "opportunity": "Low-activity customer engagement",
        "proposed_change": "Improve product discovery",
        "primary_metric": "Eligible user purchase conversion",
        "expected_direction": "Increase"
    }
])

print("========== PROPOSED BUSINESS HYPOTHESES ==========")

display(business_hypotheses)

print("\n========== VALIDATION ==========")

assert len(business_hypotheses) == 3
assert business_hypotheses["hypothesis_id"].is_unique
assert business_hypotheses["primary_metric"].notna().all()

print("Hypothesis count: PASSED")
print("Unique hypothesis IDs: PASSED")
print("Primary metric definitions: PRESENT")

print("\n========== METHODOLOGY ==========")

print("These are proposed interventions.")
print("No randomized experiments have been conducted.")
print("Observed associations do not establish causation.")
print("Experiment eligibility must be defined before randomization.")

print("\nBUSINESS HYPOTHESIS DEVELOPMENT COMPLETE")


========== PROPOSED BUSINESS HYPOTHESES ==========


,hypothesis_id,opportunity,proposed_change,primary_metric,expected_direction
0,H1,Low purchase participation,Simplify add-to-cart experience,Eligible user purchase conversion,Increase
1,H2,Cart-to-purchase friction,Simplify checkout flow,Eligible cart-user purchase conversion,Increase
2,H3,Low-activity customer engagement,Improve product discovery,Eligible user purchase conversion,Increase



========== VALIDATION ==========
Hypothesis count: PASSED
Unique hypothesis IDs: PASSED
Primary metric definitions: PRESENT

========== METHODOLOGY ==========
These are proposed interventions.
No randomized experiments have been conducted.
Observed associations do not establish causation.
Experiment eligibility must be defined before randomization.

BUSINESS HYPOTHESIS DEVELOPMENT COMPLETE


In [15]:

# Step 06.18 — A/B Experiment Design
# Experiment H1: Simplify Add-to-Cart Experience

experiment_design = {
    "experiment_id": "EXP_H1_001",
    "hypothesis_id": "H1",
    "experiment_name": "Simplified Add-to-Cart Experience",

    "objective": (
        "Evaluate whether a simplified add-to-cart "
        "experience increases purchase conversion."
    ),

    "unit_of_randomization": "Unique customer",

    "eligible_population": (
        "Customers eligible for the add-to-cart "
        "interface, defined before randomization."
    ),

    "control_group": "Existing add-to-cart interface",

    "treatment_group": "Simplified add-to-cart interface",

    "allocation": "50% control / 50% treatment",

    "primary_metric": (
        "Customers with at least one purchase "
        "within a predefined attribution window "
        "divided by randomized eligible customers."
    ),

    "null_hypothesis": (
        "Treatment and control purchase "
        "conversion rates are equal."
    ),

    "alternative_hypothesis": (
        "Treatment and control purchase "
        "conversion rates differ."
    ),

    "significance_level": 0.05,

    "target_power": 0.80,

    "test_type": "Two-sided",

    "experiment_status": "Proposed — Not conducted"
}

print("========== A/B EXPERIMENT DESIGN ==========")

for key, value in experiment_design.items():
    print(f"{key}: {value}")

print("\n========== VALIDATION ==========")

assert experiment_design["hypothesis_id"] == "H1"
assert experiment_design["significance_level"] == 0.05
assert experiment_design["target_power"] == 0.80
assert experiment_design["unit_of_randomization"] == "Unique customer"

print("Hypothesis linkage: PASSED")
print("Statistical settings: PASSED")
print("Randomization unit: PASSED")

print("\n========== IMPORTANT LIMITATIONS ==========")

print("This is a proposed experiment only.")
print("No treatment or control data exists yet.")
print("Eligibility and attribution window require final specification.")
print("Sample size and runtime must be calculated before launch.")

print("\nA/B EXPERIMENT DESIGN COMPLETE")


========== A/B EXPERIMENT DESIGN ==========
experiment_id: EXP_H1_001
hypothesis_id: H1
experiment_name: Simplified Add-to-Cart Experience
objective: Evaluate whether a simplified add-to-cart experience increases purchase conversion.
unit_of_randomization: Unique customer
eligible_population: Customers eligible for the add-to-cart interface, defined before randomization.
control_group: Existing add-to-cart interface
treatment_group: Simplified add-to-cart interface
allocation: 50% control / 50% treatment
primary_metric: Customers with at least one purchase within a predefined attribution window divided by randomized eligible customers.
null_hypothesis: Treatment and control purchase conversion rates are equal.
alternative_hypothesis: Treatment and control purchase conversion rates differ.
significance_level: 0.05
target_power: 0.8
test_type: Two-sided
experiment_status: Proposed — Not conducted

========== VALIDATION ==========
Hypothesis linkage: PASSED
Statistical settings: PASSED
Ra

In [16]:

# Step 06.19 — Baseline and MDE Planning
# These are planning assumptions, not experimental results.

baseline_rate = purchasing_customers / total_customers

relative_mde = 0.10  # Assumed 10% relative improvement

target_treatment_rate = baseline_rate * (1 + relative_mde)

absolute_mde_pp = (
    target_treatment_rate - baseline_rate
) * 100

print("========== A/B TEST PLANNING ASSUMPTIONS ==========")

print(f"Illustrative baseline: {baseline_rate * 100:.2f}%")
print(f"Relative MDE: {relative_mde * 100:.1f}%")

print(
    f"Target treatment rate: "
    f"{target_treatment_rate * 100:.2f}%"
)

print(
    f"Absolute MDE: {absolute_mde_pp:.2f} "
    "percentage points"
)

print("\n========== VALIDATION ==========")

assert 0 < baseline_rate < 1
assert 0 < target_treatment_rate < 1
assert target_treatment_rate > baseline_rate
assert relative_mde > 0

print("Baseline rate: PASSED")
print("Target rate: PASSED")
print("MDE assumptions: PASSED")

print("\n========== METHODOLOGY ==========")

print("Baseline is a historical purchase-participation proxy.")
print("It is NOT a verified experiment-eligible conversion rate.")
print("The 10% relative MDE is an illustrative assumption.")
print("Actual planning requires business and product validation.")

print("\nBASELINE AND MDE PLANNING COMPLETE")


========== A/B TEST PLANNING ASSUMPTIONS ==========
Illustrative baseline: 11.65%
Relative MDE: 10.0%
Target treatment rate: 12.81%
Absolute MDE: 1.16 percentage points

========== VALIDATION ==========
Baseline rate: PASSED
Target rate: PASSED
MDE assumptions: PASSED

========== METHODOLOGY ==========
Baseline is a historical purchase-participation proxy.
It is NOT a verified experiment-eligible conversion rate.
The 10% relative MDE is an illustrative assumption.
Actual planning requires business and product validation.

BASELINE AND MDE PLANNING COMPLETE


In [17]:

# Step 06.20 — A/B Test Sample Size Calculation
# Two independent proportions, equal allocation.

import math

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Reuse validated planning assumptions.
p_control = baseline_rate
p_treatment = target_treatment_rate

alpha = experiment_design["significance_level"]
power = experiment_design["target_power"]

# Cohen's h for two proportions.
effect_size_h = abs(
    proportion_effectsize(p_control, p_treatment)
)

# Calculate sample size per group.
power_analysis = NormalIndPower()

sample_size_raw = power_analysis.solve_power(
    effect_size=effect_size_h,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided"
)

sample_size_per_group = math.ceil(sample_size_raw)

total_sample_size = sample_size_per_group * 2

print("========== A/B TEST SAMPLE SIZE ==========")

print(f"Control baseline: {p_control * 100:.2f}%")
print(f"Treatment target: {p_treatment * 100:.2f}%")
print(f"Cohen's h: {effect_size_h:.6f}")

print("\n========== REQUIRED SAMPLE ==========")

print(f"Control customers: {sample_size_per_group:,}")
print(f"Treatment customers: {sample_size_per_group:,}")
print(f"Total customers: {total_sample_size:,}")

print("\n========== STATISTICAL SETTINGS ==========")

print(f"Significance level: {alpha}")
print(f"Target power: {power}")
print("Allocation: 50/50")
print("Test: Two-sided")

print("\n========== VALIDATION ==========")

assert sample_size_per_group > 0
assert total_sample_size == sample_size_per_group * 2
assert 0 < effect_size_h < 1

print("Sample size: PASSED")
print("Allocation consistency: PASSED")
print("Effect size: PASSED")

print("\n========== LIMITATIONS ==========")

print("Sample size is based on illustrative assumptions.")
print("Eligibility, baseline, MDE, and runtime require validation.")
print("No randomized experiment has been conducted.")

print("\nSAMPLE SIZE CALCULATION COMPLETE")


========== A/B TEST SAMPLE SIZE ==========
Control baseline: 11.65%
Treatment target: 12.81%
Cohen's h: 0.035563

========== REQUIRED SAMPLE ==========
Control customers: 12,412
Treatment customers: 12,412
Total customers: 24,824

========== STATISTICAL SETTINGS ==========
Significance level: 0.05
Target power: 0.8
Allocation: 50/50
Test: Two-sided

========== VALIDATION ==========
Sample size: PASSED
Allocation consistency: PASSED
Effect size: PASSED

========== LIMITATIONS ==========
Sample size is based on illustrative assumptions.
Eligibility, baseline, MDE, and runtime require validation.
No randomized experiment has been conducted.

SAMPLE SIZE CALCULATION COMPLETE


In [18]:

# Step 06.21 — Experiment Feasibility and Runtime Planning

import math
import pandas as pd

# Reuse verified Step 06.20 results.
required_customers = total_sample_size

# Hypothetical eligible customers per day.
# These are scenarios, NOT measured traffic.
daily_traffic_scenarios = [100, 250, 500, 1000, 2000]

runtime_results = []

for daily_customers in daily_traffic_scenarios:

    required_days = math.ceil(
        required_customers / daily_customers
    )

    required_weeks = required_days / 7

    runtime_results.append({
        "daily_eligible_customers": daily_customers,
        "required_customers": required_customers,
        "estimated_days": required_days,
        "estimated_weeks": round(required_weeks, 2)
    })

runtime_df = pd.DataFrame(runtime_results)

print("========== EXPERIMENT RUNTIME PLANNING ==========")

display(runtime_df)

print("\n========== VALIDATION ==========")

assert required_customers == 24824

assert all(
    runtime_df["estimated_days"] > 0
)

assert (
    runtime_df["estimated_days"]
    .is_monotonic_decreasing
)

print("Required sample size: PASSED")
print("Positive runtime estimates: PASSED")
print("Traffic-runtime relationship: PASSED")

print("\n========== METHODOLOGY ==========")

print("Traffic values are hypothetical scenarios.")
print("Actual eligible traffic has not been measured.")
print("Estimates assume constant daily eligible traffic.")
print("Recruitment time excludes follow-up and analysis.")
print("Experiment runtime requires product validation.")

print("\nEXPERIMENT RUNTIME PLANNING COMPLETE")


========== EXPERIMENT RUNTIME PLANNING ==========


,daily_eligible_customers,required_customers,estimated_days,estimated_weeks
0,100,24824,249,35.57
1,250,24824,100,14.29
2,500,24824,50,7.14
3,1000,24824,25,3.57
4,2000,24824,13,1.86



========== VALIDATION ==========
Required sample size: PASSED
Positive runtime estimates: PASSED
Traffic-runtime relationship: PASSED

========== METHODOLOGY ==========
Traffic values are hypothetical scenarios.
Actual eligible traffic has not been measured.
Estimates assume constant daily eligible traffic.
Recruitment time excludes follow-up and analysis.
Experiment runtime requires product validation.

EXPERIMENT RUNTIME PLANNING COMPLETE


In [19]:

# Step 06.22 — Experiment Feasibility Assessment

import math
import pandas as pd

# Reuse the validated sample size from Step 06.20.
required_customers = total_sample_size

# Hypothetical experiment recruitment windows.
planning_windows = [14, 28, 56]

feasibility_results = []

for days in planning_windows:

    minimum_daily_traffic = math.ceil(
        required_customers / days
    )

    feasibility_results.append({
        "planning_window_days": days,
        "required_customers": required_customers,
        "minimum_daily_eligible_customers": minimum_daily_traffic
    })

feasibility_df = pd.DataFrame(feasibility_results)

print("========== EXPERIMENT FEASIBILITY ==========")

display(feasibility_df)

print("\n========== VALIDATION ==========")

assert required_customers == 24824

assert all(
    feasibility_df["minimum_daily_eligible_customers"] > 0
)

assert (
    feasibility_df["minimum_daily_eligible_customers"]
    .is_monotonic_decreasing
)

print("Required sample size: PASSED")
print("Positive traffic requirements: PASSED")
print("Planning window consistency: PASSED")

print("\n========== METHODOLOGY ==========")

print("Planning windows are hypothetical.")
print("Traffic requirements assume constant recruitment.")
print("Actual experiment eligibility must be validated.")
print("Recruitment estimates exclude outcome follow-up.")
print("No randomized experiment has been conducted.")

print("\nEXPERIMENT FEASIBILITY ASSESSMENT COMPLETE")


========== EXPERIMENT FEASIBILITY ==========


,planning_window_days,required_customers,minimum_daily_eligible_customers
0,14,24824,1774
1,28,24824,887
2,56,24824,444



========== VALIDATION ==========
Required sample size: PASSED
Positive traffic requirements: PASSED
Planning window consistency: PASSED

========== METHODOLOGY ==========
Planning windows are hypothetical.
Traffic requirements assume constant recruitment.
Actual experiment eligibility must be validated.
Recruitment estimates exclude outcome follow-up.
No randomized experiment has been conducted.

EXPERIMENT FEASIBILITY ASSESSMENT COMPLETE


In [20]:

# Step 06.23 — A/B Experiment Guardrail Metrics

import pandas as pd

guardrail_metrics = pd.DataFrame([
    {
        "metric": "Purchase conversion rate",
        "metric_type": "Primary",
        "purpose": "Measure experiment success",
        "expected_direction": "Increase"
    },
    {
        "metric": "Add-to-cart rate",
        "metric_type": "Secondary",
        "purpose": "Measure cart engagement",
        "expected_direction": "Increase"
    },
    {
        "metric": "Cart-to-purchase conversion",
        "metric_type": "Guardrail",
        "purpose": "Monitor checkout performance",
        "expected_direction": "No material decrease"
    },
    {
        "metric": "Revenue per eligible customer",
        "metric_type": "Guardrail",
        "purpose": "Monitor revenue performance",
        "expected_direction": "No material decrease"
    },
    {
        "metric": "Error rate",
        "metric_type": "Guardrail",
        "purpose": "Monitor technical reliability",
        "expected_direction": "No material increase"
    }
])

print("========== EXPERIMENT METRICS ==========")

display(guardrail_metrics)

print("\n========== VALIDATION ==========")

assert len(guardrail_metrics) == 5

assert (
    guardrail_metrics["metric_type"] == "Primary"
).sum() == 1

assert (
    guardrail_metrics["metric_type"] == "Guardrail"
).sum() == 3

assert guardrail_metrics["metric"].is_unique

print("Metric count: PASSED")
print("Primary metric: PASSED")
print("Guardrail metrics: PASSED")
print("Unique metric definitions: PASSED")

print("\n========== METHODOLOGY ==========")

print("Metrics are proposed for a future experiment.")
print("Eligibility and attribution must be finalized.")
print("Guardrail thresholds require business approval.")
print("Technical error data may require new instrumentation.")
print("No randomized experiment has been conducted.")

print("\nEXPERIMENT GUARDRAIL METRICS COMPLETE")


========== EXPERIMENT METRICS ==========


,metric,metric_type,purpose,expected_direction
0,Purchase conversion rate,Primary,Measure experiment success,Increase
1,Add-to-cart rate,Secondary,Measure cart engagement,Increase
2,Cart-to-purchase conversion,Guardrail,Monitor checkout performance,No material decrease
3,Revenue per eligible customer,Guardrail,Monitor revenue performance,No material decrease
4,Error rate,Guardrail,Monitor technical reliability,No material increase



========== VALIDATION ==========
Metric count: PASSED
Primary metric: PASSED
Guardrail metrics: PASSED
Unique metric definitions: PASSED

========== METHODOLOGY ==========
Metrics are proposed for a future experiment.
Eligibility and attribution must be finalized.
Guardrail thresholds require business approval.
Technical error data may require new instrumentation.
No randomized experiment has been conducted.

EXPERIMENT GUARDRAIL METRICS COMPLETE


In [21]:

# Step 06.24 — Experiment Decision Framework

import pandas as pd

decision_framework = pd.DataFrame([
    {
        "criterion": "Data quality",
        "requirement": "Valid assignment, tracking, and eligibility",
        "decision_role": "Mandatory"
    },
    {
        "criterion": "Sample size",
        "requirement": "Reach planned sample size",
        "decision_role": "Mandatory"
    },
    {
        "criterion": "Statistical significance",
        "requirement": "Two-sided p-value below 0.05",
        "decision_role": "Statistical evidence"
    },
    {
        "criterion": "Practical significance",
        "requirement": "Evaluate lift against business-defined MDE",
        "decision_role": "Business evaluation"
    },
    {
        "criterion": "Guardrail metrics",
        "requirement": "No unacceptable guardrail deterioration",
        "decision_role": "Safety check"
    },
    {
        "criterion": "Experiment integrity",
        "requirement": "No major randomization or tracking issues",
        "decision_role": "Mandatory"
    }
])

print("========== EXPERIMENT DECISION FRAMEWORK ==========")

display(decision_framework)

print("\n========== VALIDATION ==========")

assert len(decision_framework) == 6

assert decision_framework["criterion"].is_unique

assert (
    decision_framework["decision_role"] == "Mandatory"
).sum() == 3

assert "Guardrail metrics" in decision_framework["criterion"].values

print("Decision criteria count: PASSED")
print("Unique criteria: PASSED")
print("Mandatory checks: PASSED")
print("Guardrail inclusion: PASSED")

print("\n========== DECISION PRINCIPLES ==========")

print("Statistical significance alone does not justify rollout.")
print("Practical impact and guardrails must be reviewed.")
print("Confidence intervals should accompany p-values.")
print("Final decisions require business and product review.")
print("This framework is proposed, not experimentally validated.")

print("\nEXPERIMENT DECISION FRAMEWORK COMPLETE")


========== EXPERIMENT DECISION FRAMEWORK ==========


,criterion,requirement,decision_role
0,Data quality,"Valid assignment, tracking, and eligibility",Mandatory
1,Sample size,Reach planned sample size,Mandatory
2,Statistical significance,Two-sided p-value below 0.05,Statistical evidence
3,Practical significance,Evaluate lift against business-defined MDE,Business evaluation
4,Guardrail metrics,No unacceptable guardrail deterioration,Safety check
5,Experiment integrity,No major randomization or tracking issues,Mandatory



========== VALIDATION ==========
Decision criteria count: PASSED
Unique criteria: PASSED
Mandatory checks: PASSED
Guardrail inclusion: PASSED

========== DECISION PRINCIPLES ==========
Statistical significance alone does not justify rollout.
Practical impact and guardrails must be reviewed.
Confidence intervals should accompany p-values.
Final decisions require business and product review.
This framework is proposed, not experimentally validated.

EXPERIMENT DECISION FRAMEWORK COMPLETE


In [22]:

# Step 06.25 — Experiment Risks and Mitigation Plan

import pandas as pd

experiment_risks = pd.DataFrame([
    {
        "risk_id": "R1",
        "risk": "Sample ratio mismatch",
        "impact": "Unequal or unexpected experiment allocation",
        "mitigation": "Monitor assignment counts and investigate mismatches"
    },
    {
        "risk_id": "R2",
        "risk": "Tracking failures",
        "impact": "Missing or incorrect conversion events",
        "mitigation": "Validate event tracking before and during experiment"
    },
    {
        "risk_id": "R3",
        "risk": "Premature stopping",
        "impact": "Inflated false-positive risk",
        "mitigation": "Use predefined stopping and analysis rules"
    },
    {
        "risk_id": "R4",
        "risk": "Customer contamination",
        "impact": "Customers exposed to both variants",
        "mitigation": "Maintain persistent customer-level assignment"
    },
    {
        "risk_id": "R5",
        "risk": "Seasonality",
        "impact": "Time-dependent changes in customer behavior",
        "mitigation": "Run variants concurrently across representative periods"
    },
    {
        "risk_id": "R6",
        "risk": "Guardrail deterioration",
        "impact": "Conversion gains accompanied by negative outcomes",
        "mitigation": "Monitor guardrails using predefined review rules"
    }
])

print("========== EXPERIMENT RISKS ==========")

display(experiment_risks)

print("\n========== VALIDATION ==========")

assert len(experiment_risks) == 6

assert experiment_risks["risk_id"].is_unique

assert experiment_risks["mitigation"].notna().all()

assert (
    experiment_risks["mitigation"].str.strip().str.len() > 0
).all()

print("Risk count: PASSED")
print("Unique risk IDs: PASSED")
print("Mitigation completeness: PASSED")
print("Non-empty mitigation plans: PASSED")

print("\n========== METHODOLOGY ==========")

print("Risks are identified for a proposed experiment.")
print("Mitigation plans require operational implementation.")
print("Monitoring thresholds must be defined before launch.")
print("No randomized experiment has been conducted.")

print("\nEXPERIMENT RISK ASSESSMENT COMPLETE")


========== EXPERIMENT RISKS ==========


,risk_id,risk,impact,mitigation
0,R1,Sample ratio mismatch,Unequal or unexpected experiment allocation,Monitor assignment counts and investigate mism...
1,R2,Tracking failures,Missing or incorrect conversion events,Validate event tracking before and during expe...
2,R3,Premature stopping,Inflated false-positive risk,Use predefined stopping and analysis rules
3,R4,Customer contamination,Customers exposed to both variants,Maintain persistent customer-level assignment
4,R5,Seasonality,Time-dependent changes in customer behavior,Run variants concurrently across representativ...
5,R6,Guardrail deterioration,Conversion gains accompanied by negative outcomes,Monitor guardrails using predefined review rules



========== VALIDATION ==========
Risk count: PASSED
Unique risk IDs: PASSED
Mitigation completeness: PASSED
Non-empty mitigation plans: PASSED

========== METHODOLOGY ==========
Risks are identified for a proposed experiment.
Mitigation plans require operational implementation.
Monitoring thresholds must be defined before launch.
No randomized experiment has been conducted.

EXPERIMENT RISK ASSESSMENT COMPLETE


In [25]:

# Step 06.26B — Verify Existing Segment DataFrames

print("========== SEGMENT STATS ==========")
print("Columns:", segment_stats.columns.tolist())
display(segment_stats.head())

print("\n========== SEGMENT CONFIDENCE INTERVALS ==========")
print("Columns:", segment_ci.columns.tolist())
display(segment_ci.head())

print("\nSEGMENT DATAFRAME INSPECTION COMPLETE")


========== SEGMENT STATS ==========
Columns: ['activity_segment', 'total_customers', 'purchasing_customers', 'participation_rate']


,activity_segment,total_customers,purchasing_customers,participation_rate
0,Low Activity,30626,657,2.145236
1,Moderate Activity,15371,1932,12.569124
2,High Activity,11902,3032,25.474710
3,Very High Activity,3040,1478,48.618421



========== SEGMENT CONFIDENCE INTERVALS ==========
Columns: ['activity_segment', 'total_customers', 'purchasing_customers', 'participation_rate', 'ci_lower_pct', 'ci_upper_pct']


,activity_segment,total_customers,purchasing_customers,participation_rate,ci_lower_pct,ci_upper_pct
0,Low Activity,30626,657,2.145236,1.988870,2.313606
1,Moderate Activity,15371,1932,12.569124,12.054397,13.102555
2,High Activity,11902,3032,25.474710,24.699921,26.265326
3,Very High Activity,3040,1478,48.618421,46.844580,50.395750



SEGMENT DATAFRAME INSPECTION COMPLETE


In [27]:

# Step 06.26 — Final Statistical Analysis Summary

print("========== PHASE 06 FINAL SUMMARY ==========")

print("\n1. CUSTOMER PURCHASE PARTICIPATION")
print("Total customers: 60,939")
print("Purchasing customers: 7,099")
print("Observed purchase participation: 11.65%")
print("95% Wilson confidence interval: 11.40% - 11.91%")

print("\n2. CUSTOMER ACTIVITY SEGMENTS")
display(
    segment_ci[
        [
            "activity_segment",
            "total_customers",
            "purchasing_customers",
            "participation_rate"
        ]
    ]
)

print("\n3. STATISTICAL ASSOCIATION")
print("Chi-square statistic: 8947.6337")
print("Degrees of freedom: 3")
print("Cramer's V: 0.3832")
print("Interpretation: Association, not causation.")

print("\n4. SEGMENT COMPARISON")
print("Low Activity purchase participation: 2.15%")
print("Very High Activity purchase participation: 48.62%")
print("Observed difference: 46.47 percentage points")
print("95% difference CI: 44.69 - 48.26 percentage points")

print("\n5. PROPOSED A/B EXPERIMENT")
print("Experiment: Simplified Add-to-Cart Experience")
print("Randomization unit: Unique customer")
print("Allocation: 50% control / 50% treatment")
print("Significance level: 0.05")
print("Target power: 80%")
print("Illustrative relative MDE: 10%")

print("\n6. ILLUSTRATIVE SAMPLE SIZE")
print("Control customers: 12,412")
print("Treatment customers: 12,412")
print("Total required customers: 24,824")

print("\n7. EXPERIMENT GOVERNANCE")
print(f"Documented risks: {len(experiment_risks)}")
print("Guardrail metrics: Defined")
print("Decision framework: Defined")
print("Experiment status: Proposed - Not conducted")

print("\n========== FINAL VALIDATION ==========")

assert total_customers == 60939
assert purchasing_customers == 7099
assert len(experiment_risks) == 6
assert len(segment_ci) == 4

print("Customer totals: PASSED")
print("Segment coverage: PASSED")
print("Risk documentation: PASSED")

print("\nIMPORTANT LIMITATIONS")
print("- Historical observations do not establish causality.")
print("- Population inference depends on sampling assumptions.")
print("- A/B baseline and MDE are illustrative.")
print("- Experiment eligibility and tracking require validation.")
print("- No randomized experiment has been conducted.")

print("\nPHASE 06 STATISTICAL SUMMARY COMPLETE")


========== PHASE 06 FINAL SUMMARY ==========

1. CUSTOMER PURCHASE PARTICIPATION
Total customers: 60,939
Purchasing customers: 7,099
Observed purchase participation: 11.65%
95% Wilson confidence interval: 11.40% - 11.91%

2. CUSTOMER ACTIVITY SEGMENTS


,activity_segment,total_customers,purchasing_customers,participation_rate
0,Low Activity,30626,657,2.145236
1,Moderate Activity,15371,1932,12.569124
2,High Activity,11902,3032,25.474710
3,Very High Activity,3040,1478,48.618421



3. STATISTICAL ASSOCIATION
Chi-square statistic: 8947.6337
Degrees of freedom: 3
Cramer's V: 0.3832
Interpretation: Association, not causation.

4. SEGMENT COMPARISON
Low Activity purchase participation: 2.15%
Very High Activity purchase participation: 48.62%
Observed difference: 46.47 percentage points
95% difference CI: 44.69 - 48.26 percentage points

5. PROPOSED A/B EXPERIMENT
Experiment: Simplified Add-to-Cart Experience
Randomization unit: Unique customer
Allocation: 50% control / 50% treatment
Significance level: 0.05
Target power: 80%
Illustrative relative MDE: 10%

6. ILLUSTRATIVE SAMPLE SIZE
Control customers: 12,412
Treatment customers: 12,412
Total required customers: 24,824

7. EXPERIMENT GOVERNANCE
Documented risks: 6
Guardrail metrics: Defined
Decision framework: Defined
Experiment status: Proposed - Not conducted

========== FINAL VALIDATION ==========
Customer totals: PASSED
Segment coverage: PASSED
Risk documentation: PASSED

IMPORTANT LIMITATIONS
- Historical observ